# MACE Foundation Models for Materials Simulation

This notebook introduces **machine-learned interatomic potentials (MLIPs)**, focusing on MACE-MP-0 — a universal foundation model for atomistic simulation. We use MACE for fast geometry relaxation before running more expensive DFT calculations.

**Topics covered:**
- What are MLIPs and why use them?
- MACE-MP-0: architecture and training
- Using MACE through ASE
- Limitations: no charge, no magnetism, no electronic structure
- MACE as a pre-relaxation tool for DFT

---


## 1. The Fidelity Landscape: From Classical to Quantum

Atomistic simulation methods span a wide range of **computational cost vs accuracy** — what is sometimes called the **fidelity landscape**:

| Method | Physical basis | Accuracy | Cost (relative) |
|--------|---------------|----------|----------------|
| Classical potentials | Empirical rules | Low | 1x |
| **MLIP (MACE)** | Learned from DFT | ~DFT | 100-10,000x |
| DFT (PBE) | Quantum mechanics | Medium | Reference |
| DFT (HSE06) | Quantum mechanics | High | 10-100x DFT |
| GW / BSE | Many-body theory | Very high | 1000x DFT |

The key insight is that **different questions need different fidelity**:
- Geometry relaxation: MLIP accuracy is sufficient
- Band gap: need at least HSE06
- Optical spectra: need GW+BSE

This **multi-fidelity** approach — using cheaper methods where they are adequate, and expensive methods only where necessary — is now standard practice in computational materials science.

### Foundation models

Traditional MLIPs are trained for specific systems (e.g. silicon only, requiring new training data for every new material). **Foundation models** change this: trained on vast, diverse datasets, they generalise across the periodic table with no retraining required.

**MACE-MP-0** is the leading foundation model for inorganic materials:
- Trained on the entire Materials Project (~150,000 structures, 89 elements)
- Works for almost any material out of the box
- Achieves near-DFT accuracy for geometry and energetics
- Downloads automatically on first use (~50 MB)
- Available in `small`, `medium`, and `large` variants (`medium` recommended)

Other foundation models you may encounter:
- **CHGNet** — also trained on Materials Project, includes magnetic moments
- **SevenNet** — strong for bulk properties
- **ORB** — fast inference, good for molecular systems
- **MACE-OFF** — variant for organic molecules

For this course we use MACE-MP-0 (`medium`) as our default geometry engine.


In [ ]:
# Check MACE is installed
try:
    from mace.calculators import mace_mp
    print("MACE is available")
except ImportError:
    print("MACE not installed. Run: pip install mace-torch")

# The torch.compiler fix needed on Intel Mac
import torch
if not hasattr(torch.compiler, 'is_compiling'):
    torch.compiler.is_compiling = lambda: False
    print("Applied torch.compiler fix for Intel Mac")


## 2. Using MACE Through ASE

MACE integrates with ASE as a standard calculator. The key function is `mace_mp()` which downloads and caches the pre-trained model automatically.


In [ ]:
from mace.calculators import mace_mp
from ase.build import bulk
import numpy as np

# Load the MACE-MP-0 calculator
# model="medium" is the recommended default — good balance of speed and accuracy
# dispersion=False: van der Waals correction (use True for layered materials)
# default_dtype="float64": slower but more accurate, recommended for geometry opt
calc = mace_mp(model="medium", dispersion=False, default_dtype="float64")

# Build a simple structure and compute energy
diamond = bulk('C', 'diamond', a=3.57)
diamond.calc = calc

energy = diamond.get_potential_energy()
forces = diamond.get_forces()

print(f"Diamond total energy: {energy:.4f} eV")
print(f"Max force (should be ~0 at equilibrium): {np.max(np.abs(forces)):.6f} eV/Å")


## 3. Multi-Fidelity Geometry Relaxation

A key strategy in modern computational materials science is **multi-fidelity optimisation**: use a cheap, approximate method to get close to the minimum, then refine with a more expensive method only when needed.

For defect calculations:
- **MACE** handles the heavy lifting of ionic relaxation — finding approximately correct atomic positions in seconds
- **GPAW** then does a single-point electronic structure calculation on the pre-relaxed geometry

This is valid because:
1. MACE geometry errors are typically <1% in bond lengths — within DFT accuracy anyway
2. The GPAW SCF converges faster from a good starting geometry
3. The electronic structure (DOS, defect levels) is not sensitive to sub-percent geometry errors

The alternative — full DFT relaxation — would take 1-2 hours for a 60-atom supercell on a single core. MACE reduces this to ~30 seconds.


In [ ]:
from ase.optimize import BFGS
from ase.build import bulk, make_supercell
from ase.geometry import get_distances
import time

# Build a slightly distorted diamond supercell
prim = bulk('C', 'diamond', a=3.57)
sc = make_supercell(prim, np.diag([2, 2, 2]))

# Add random displacement to simulate an unrelaxed structure
np.random.seed(42)
sc.positions += np.random.uniform(-0.2, 0.2, sc.positions.shape)

sc.calc = mace_mp(model="medium", dispersion=False, default_dtype="float64")

print(f"Initial max force: {np.max(np.abs(sc.get_forces())):.4f} eV/Å")

t0 = time.time()
opt = BFGS(sc, logfile=None)
opt.run(fmax=0.01)
t1 = time.time()

print(f"Final max force:   {np.max(np.abs(sc.get_forces())):.4f} eV/Å")
print(f"Relaxation time:   {t1-t0:.1f} seconds")
print(f"\nFor comparison, GPAW DFT relaxation of same system: ~15-60 minutes")


## 4. MACE for Defect Structures

MACE is particularly useful for defect pre-relaxation. It captures the local distortion around the defect well, giving a good starting geometry for DFT.


In [ ]:
# Relax an NV centre with MACE
from ase.build import bulk, make_supercell
from ase.geometry import get_distances
from ase.optimize import BFGS
import numpy as np

# Build NV defect
prim = bulk('C', 'diamond', a=3.57)
sc = make_supercell(prim, np.diag([3, 3, 3]))

_, D = get_distances(sc.positions, sc.positions, cell=sc.cell, pbc=True)
np.fill_diagonal(D, np.inf)
vac_idx = int(np.argmin(D[0]))

sym = sc.get_chemical_symbols()
sym[0] = 'N'
sc.set_chemical_symbols(sym)

# Store pristine neighbour positions for comparison
neighbour_indices = np.argsort(D[0])[:4].tolist()
# Remove vacancy index
neighbour_indices = [i for i in neighbour_indices if i != vac_idx][:3]
pristine_positions = sc.positions[neighbour_indices].copy()

del sc[vac_idx]
# Update neighbour indices after deletion
neighbour_indices_updated = [i if i < vac_idx else i-1 for i in neighbour_indices]

# Random displacement
np.random.seed(42)
sc.positions += np.random.uniform(-0.1, 0.1, sc.positions.shape)

sc.calc = mace_mp(model="medium", dispersion=False, default_dtype="float64")
opt = BFGS(sc, logfile=None)
opt.run(fmax=0.01)

# Measure inward relaxation of C neighbours toward vacancy
print("C neighbour displacements after MACE relaxation:")
print("(negative = moved toward vacancy)")
for idx in neighbour_indices_updated:
    relaxed_pos = sc.positions[idx]
    # Approximate: compare to ideal lattice position
    print(f"  Atom {idx}: position {relaxed_pos.round(3)}")

print(f"\nMax residual force: {np.max(np.abs(sc.get_forces())):.4f} eV/Å")
print("\nThis geometry is ready to pass to GPAW for electronic structure.")


## 5. Limitations of MACE

MACE is powerful but has important limitations for quantum optics applications:

### ❌ No charge states
MACE cannot distinguish NV(-) from NV0 — it has no concept of electron count or charge state. The geometry of charged defects may differ from neutral, so MACE relaxation is an approximation.

### ❌ No magnetism
MACE has no spin — it cannot predict magnetic moments or find Jahn-Teller distortions that depend on spin state. For spin-1 systems like NV(-) or V_B, the relaxation may miss symmetry-breaking distortions.

### ❌ No electronic structure
MACE gives you energy and forces only — no band structure, no DOS, no defect levels. You always need DFT for the electronic structure.

### [yes] When MACE is good enough
For **geometry pre-relaxation** — getting approximate atomic positions before DFT — MACE is excellent. The geometry error is typically <1% for bond lengths, which is within DFT accuracy anyway.

### Workflow summary


In [ ]:
workflow = '''
Multi-fidelity workflow for defect electronic structure:
=========================================================

STAGE 1 — Structure (MACE, ~30 seconds)
  Build defect supercell
  Random displacement (break symmetry)
  MACE relaxation (fmax=0.01 eV/A)
  --> Good geometry, no electronic structure

STAGE 2 — Electronics (GPAW, ~15-60 minutes)
  Set charge state and magnetic moments
  GPAW SCF with dzp basis
  DOS / band structure analysis
  --> Defect levels, spin state, gap

Why split it this way?
  - MACE is 1000x faster than DFT for forces
  - Geometry accuracy from MACE is sufficient for DFT input
  - DFT is only needed for electronic structure (MACE has none)
  - Total time: 1 hour instead of 10+ hours

This is the multi-fidelity principle: match method fidelity
to the question being asked.
'''
print(workflow)


## 6. Equation of State with MACE

MACE can also be used for quick structural screening — for example, computing the equation of state to find equilibrium lattice constants.


In [ ]:
from ase.eos import EquationOfState

# EOS for diamond using MACE
prim = bulk('C', 'diamond', a=3.57)
volumes, energies = [], []

for scale in np.linspace(0.94, 1.06, 10):
    sc_scaled = prim.copy()
    sc_scaled.set_cell(prim.cell * scale, scale_atoms=True)
    sc_scaled.calc = mace_mp(model="medium", dispersion=False, default_dtype="float64")
    volumes.append(sc_scaled.get_volume())
    energies.append(sc_scaled.get_potential_energy())

eos = EquationOfState(volumes, energies, eos='birchmurnaghan')
v0, e0, B = eos.fit()

print(f"MACE equilibrium volume: {v0:.3f} Å³")
print(f"MACE bulk modulus: {B / 1.6022e-19 * 1e30 * 1e-9:.1f} GPa")
print(f"Experimental bulk modulus: 442 GPa")

fig = eos.plot(show=False)
plt.title('Equation of State — Diamond (MACE-MP-0)')
plt.tight_layout()
plt.show()
